# ARTI 308 – Lab 5 Tasks
## Setup and Data Loading
First, we will import the necessary libraries and load the dataset.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

# Load the dataset
DATA_PATH = "talabat_enhanced_orders.csv"
df = pd.read_csv(DATA_PATH)
df_fe = df.copy()

df_fe.head(3)

,Order_ID,User_ID,Restaurant_ID,Driver_ID,Item_Name,Quantity,Total_Price,Order_Time,Delivery_Time,Delivery_Duration_Minutes,City,Payment_Method,Order_Status,Driver_Vehicle,Restaurant_Lat,Restaurant_Lon,Customer_Lat,Customer_Lon,Driver_Lat,Driver_Lon,Delivery_Distance_km,Traffic_Level,Driver_Availability
0,1,U3522,358,485,Fried Chicken,3,273.72,2025-06-16 08:32:00,2025-06-16 09:11:00,39,Alexandria,Wallet,Delivered,Motorbike,31.195082,29.921931,31.191404,29.904982,31.215658,29.910664,1.666106,High,Offline
1,2,U9214,316,65,Sandwich,3,365.82,2025-06-03 21:27:00,2025-06-03 22:00:00,33,Zagazig,Credit Card,Delivered,Motorbike,30.605729,31.503079,30.586047,31.485820,30.580329,31.502380,2.738698,Low,Online
2,3,U7307,357,309,Koshary,3,401.94,2025-06-01 14:48:00,2025-06-01 15:26:00,38,Assiut,Cash,In Transit,Car,27.190180,31.177741,27.164869,31.169218,27.162976,31.189458,2.929079,Medium,Online


## Base Preprocessing & Helper Functions
We must recreate the basic engineered features (time, price) and the Haversine distance function from the lab so our tasks work correctly.

In [2]:
df_fe["Order_Time"] = pd.to_datetime(df_fe["Order_Time"], errors="coerce")
df_fe["order_hour"] = df_fe["Order_Time"].dt.hour
df_fe["order_dayofweek"] = df_fe["Order_Time"].dt.dayofweek
df_fe["is_weekend"] = df_fe["order_dayofweek"].isin([5,6]).astype(int)

df_fe["price_per_item"] = df_fe["Total_Price"] / df_fe["Quantity"]

def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized Haversine distance in kilometers."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

### Task 1: Create a New Engineered Feature
**Feature Name:** `driver_to_rest_distance_km`

**Justification:** Calculating the initial distance a driver has to travel to reach the restaurant can significantly impact the overall delivery time. If a driver is assigned but is far from the restaurant, the order might be delayed, increasing the likelihood of the order being cancelled. Therefore, the Haversine distance between the driver's current location and the restaurant provides a useful predictive signal for `Order_Status`.

In [3]:
df_fe['driver_to_rest_distance_km'] = haversine_km(
    df_fe['Driver_Lat'], df_fe['Driver_Lon'],
    df_fe['Restaurant_Lat'], df_fe['Restaurant_Lon']
)

df_fe[['Driver_Lat', 'Driver_Lon', 'Restaurant_Lat', 'Restaurant_Lon', 'driver_to_rest_distance_km']].head()

,Driver_Lat,Driver_Lon,Restaurant_Lat,Restaurant_Lon,driver_to_rest_distance_km
0,31.215658,29.910664,31.195082,29.921931,2.526494
1,30.580329,31.502380,30.605729,31.503079,2.825147
2,27.162976,31.189458,27.190180,31.177741,3.239299
3,31.054690,31.401187,31.041846,31.381229,2.377942
4,31.035350,31.389315,31.024141,31.376104,1.771507


### Task 2: Try a different rule for `is_peak_hour`
**Discussion:** Instead of the lab's original broad peak hours, we will define peak hours strictly as the evening dinner rush from 6 PM to 10 PM (18:00 - 22:00). Narrowing this window might help the model better capture the specific high-traffic delays that cause cancellations or delays.

In [4]:
df_fe['is_peak_hour'] = df_fe['order_hour'].isin([18, 19, 20, 21, 22]).astype(int)

print("New Peak Hour Distribution:")
print(df_fe['is_peak_hour'].value_counts())

New Peak Hour Distribution:
is_peak_hour
0    79340
1    20660
Name: count, dtype: int64


### Task 3: Change `top_k` in `Item_Name_reduced` and compare
We will loop through `top_k` values of 10, 30, and 50. For each value, we will recreate the `Item_Name_reduced` feature, prepare the data (dropping leakage columns), train a Random Forest, and output the accuracy and top feature importances.

In [5]:
top_k_values = [10, 30, 50]
target_col = "Order_Status"

drop_cols_base = [
    "Order_ID", "User_ID", "Restaurant_ID", "Driver_ID",
    "Order_Time", "Delivery_Time", "Delivery_Duration_Minutes", "Item_Name"
]

for k in top_k_values:
    print(f"--- Evaluating for top_k = {k} ---")

    top_items = df_fe["Item_Name"].value_counts().head(k).index
    df_fe["Item_Name_reduced"] = np.where(df_fe["Item_Name"].isin(top_items), df_fe["Item_Name"], "Other")

    drop_cols = [c for c in drop_cols_base if c in df_fe.columns]
    X = df_fe.drop(columns=drop_cols + [target_col])
    y = df_fe[target_col]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = X_train.select_dtypes(include=[np.number, "bool"]).columns.tolist()

    prep = ColumnTransformer(transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ])

    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced_subsample")
    model = Pipeline(steps=[("preprocess", prep), ("rf", rf)])
    model.fit(X_train, y_train)

    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"Accuracy: {acc:.4f}")

    ohe = model.named_steps["preprocess"].named_transformers_["cat"]
    cat_fn = ohe.get_feature_names_out(cat_cols) if len(cat_cols) > 0 else np.array([])
    all_fn = np.concatenate([cat_fn, np.array(num_cols)])

    imps = model.named_steps["rf"].feature_importances_
    fi = pd.DataFrame({"feature": all_fn, "importance": imps}).sort_values("importance", ascending=False)

    print("Top 3 Features:")
    print(fi.head(3).to_string(index=False))
    print("\n")

--- Evaluating for top_k = 10 ---
Accuracy: 0.8519
Top 3 Features:
                   feature  importance
driver_to_rest_distance_km    0.074901
      Delivery_Distance_km    0.074434
            price_per_item    0.072816


--- Evaluating for top_k = 30 ---
Accuracy: 0.8519
Top 3 Features:
                   feature  importance
driver_to_rest_distance_km    0.074901
      Delivery_Distance_km    0.074434
            price_per_item    0.072816


--- Evaluating for top_k = 50 ---
Accuracy: 0.8519
Top 3 Features:
                   feature  importance
driver_to_rest_distance_km    0.074901
      Delivery_Distance_km    0.074434
            price_per_item    0.072816




### Task 4: Run Feature Selection and Explain
**Explanation:** By using `SelectFromModel` with a `"median"` threshold, we instruct the pipeline to discard the bottom 50% of the least important features before training the final classifier. If the accuracy remains roughly the same (or improves slightly), this selection is highly beneficial. It results in a simpler model that trains faster, is less prone to overfitting on noisy data, and is easier to explain to business stakeholders.

In [6]:
from sklearn.feature_selection import SelectFromModel

selector = SelectFromModel(
    estimator=RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced_subsample"),
    threshold="median"
)

model_fs = Pipeline(steps=[
    ("preprocess", prep),
    ("select", selector),
    ("rf", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced_subsample"))
])

model_fs.fit(X_train, y_train)
y_pred_fs = model_fs.predict(X_test)

print("Baseline Accuracy (from top_k=50):", round(acc, 4))
print("Accuracy (with feature selection):", round(accuracy_score(y_test, y_pred_fs), 4))
print("\nClassification Report (with feature selection):")
print(classification_report(y_test, y_pred_fs))

Baseline Accuracy (from top_k=50): 0.8519
Accuracy (with feature selection): 0.8519

Classification Report (with feature selection):
              precision    recall  f1-score   support

   Cancelled       0.00      0.00      0.00      1963
   Delivered       0.85      1.00      0.92     17039
  In Transit       0.00      0.00      0.00       998

    accuracy                           0.85     20000
   macro avg       0.28      0.33      0.31     20000
weighted avg       0.73      0.85      0.78     20000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
